# RBI Data Quality Pipeline: Stage 3 (Sudden Jumps)

This check looks for weeks where a value jumps far more than usual, which can
point to a unit change, a typo or a reporting error. It measures each week's
percent change and scores how unusual it is using the median and MAD.

Later stages will add a check that the rupee and dollar columns agree.

In [1]:
import pandas as pd
data = pd.read_csv("table2_clean.csv",parse_dates=["week_ended"])
print(data.shape)

(1334, 11)


## Looking at weekly changes

Reserves are very different in size across 25 years (about 199,000 crore in
2001, about 6.8 million in 2026), so I measure percent change and not the raw
difference. A typical week moves by less than 1%, and the biggest moves are
about 5% in either direction.

In [2]:
changes = data["total_reserves_inr"].pct_change()*100
print(changes.describe())

count    1333.000000
mean        0.270103
std         0.948906
min        -4.997012
25%        -0.276925
50%         0.249007
75%         0.807240
max         5.063419
Name: total_reserves_inr, dtype: float64


To see what the largest moves look like, I listed the ten biggest weekly
changes in rupee reserves.

In [3]:
look = pd.DataFrame({"day" : data["week_ended"], "changes" : data["total_reserves_inr"].pct_change()*100})
look["size"] = look["changes"].abs()
print(look.sort_values("size",ascending=False).head(10))

            day   changes      size
1326 2026-09-04  5.063419  5.063419
247  2005-12-30 -4.997012  4.997012
338  2007-09-28  4.606336  4.606336
646  2013-08-23  3.890576  3.890576
476  2010-05-21  3.852017  3.852017
396  2008-11-07 -3.507424  3.507424
371  2008-05-16  3.410841  3.410841
99   2003-02-28 -3.407197  3.407197
987  2020-03-06  3.378527  3.378527
395  2008-10-31 -3.319125  3.319125


I added the dollar change for the same ten weeks. In five of them both
currencies moved together. In the other five, the dollar change was small.

In [4]:
look["changes_usd"] = data["total_reserves_usd"].pct_change()*100
print(look.sort_values("size",ascending=False).head(10))

            day   changes      size  changes_usd
1326 2026-09-04  5.063419  5.063419     6.061359
247  2005-12-30 -4.997012  4.997012    -4.751128
338  2007-09-28  4.606336  4.606336     5.032409
646  2013-08-23  3.890576  3.890576    -0.389265
476  2010-05-21  3.852017  3.852017     0.023417
396  2008-11-07 -3.507424  3.507424    -0.600673
371  2008-05-16  3.410841  3.410841     0.441637
99   2003-02-28 -3.407197  3.407197    -3.259454
987  2020-03-06  3.378527  3.378527     1.183097
395  2008-10-31 -3.319125  3.319125    -2.140743


In [5]:
weeks = pd.to_datetime(["2010-05-14","2010-05-21"])
two_weeks = data[data["week_ended"].isin(weeks)].copy()

two_weeks["rate"] = two_weeks["total_reserves_inr"]*10/two_weeks["total_reserves_usd"]
print(two_weeks[["week_ended","total_reserves_inr","total_reserves_usd","rate"]])
print(two_weeks["rate"].pct_change()*100)

    week_ended  total_reserves_inr  total_reserves_usd       rate
475 2010-05-14           1231355.0            273300.0  45.055068
476 2010-05-21           1278787.0            273364.0  46.779642
475         NaN
476    3.827703
Name: rate, dtype: float64


## Rupee or dollar?

In the week of 2010-05-21, rupee reserves rose 3.85% while dollar reserves
were flat, and the implied rate went from 45.06 to 46.78 rupees per dollar.
So the rupee jump came from the exchange rate, not from the reserves. A jump
check on the rupee column would flag weeks where nothing is wrong with the
data, so I run the check on the dollar columns.

## Why MAD and not standard deviation

To see the difference, I made a list of ten values near 10, then replaced one
with 100. The standard deviation jumped from 0.67 to 28.5, while the MAD only
moved from 0.0 to 0.5. An extreme value inflates the standard deviation and
can hide itself, but the MAD barely notices it. The first list also shows a
catch: when more than half the values equal the median, the MAD is 0 and a
score would divide by zero, so the function checks for that.

In [6]:
x = pd.Series([9, 10, 10, 11, 10, 9, 11, 10, 10, 10])
print(x.std())
med = x.median()
MAD = (x-med).abs().median()
print(MAD)

0.6666666666666666
0.0


In [7]:
x = pd.Series([9, 10, 10, 11, 10, 9, 11, 10, 10, 100])
print(x.std())
med = x.median()
MAD = (x-med).abs().median()
print(MAD)

28.468305963728234
0.5


## Scoring how unusual a week is

Each week gets a score: its distance from the median change, divided by the
MAD, and scaled by 0.6745 so the score reads like a z-score. A week is flagged
when its score is above 3.5, a common rule of thumb that I treat as a setting
to test later.

In [8]:
changes = data["total_reserves_usd"].pct_change()*100
changes = changes.dropna()
median_change = changes.median()
print(median_change)

0.21361225692397845


In [9]:
mad = (changes-median_change).abs().median()
print(mad)

0.4388973006368535


In [10]:
scores = 0.6745 * (changes - median_change)/mad
table = pd.DataFrame({"day":data["week_ended"], "change": changes, "score":scores})
flagged_weeks = table[table["score"].abs() > 3.5 ]
print(len(flagged_weeks))
print(flagged_weeks.sort_values("score",ascending=False))

19
            day    change     score
1326 2026-09-04  6.061359  8.986853
338  2007-09-28  5.032409  7.405556
131  2003-10-10  3.122994  4.471155
157  2004-04-09  2.990505  4.267545
191  2004-12-03  2.989214  4.265561
305  2007-02-09  2.794270  3.965971
1127 2022-11-11  2.777504  3.940204
1064 2021-08-27  2.701074  3.822746
424  2009-05-22  2.530625  3.560799
395  2008-10-31 -2.140743 -3.618187
387  2008-09-05 -2.200407 -3.709879
1329 2026-09-25 -2.394995 -4.008923
989  2020-03-20 -2.486652 -4.149783
1232 2024-11-15 -2.628761 -4.368177
391  2008-10-03 -2.699619 -4.477071
99   2003-02-28 -3.259454 -5.337429
392  2008-10-10 -3.499671 -5.706595
247  2005-12-30 -4.751128 -7.629843
394  2008-10-24 -5.648701 -9.009237


19 of 1,333 weeks are flagged. Five of them fall between September and
October 2008, all drops. The biggest jump is 2026-09-04 (score 9.0). A flag
means "look at this", not "this is wrong": I can't tell errors from real
events, because the data has no ground truth.

## The jump check as a function

The function takes a column of values and the dates, and returns the flagged
weeks. If the MAD is 0, it returns nothing. The cutoff is an input, so I can
change it later.

In [11]:
def find_jumps(values,dates,cutoff=3.5):
    changes = values.pct_change()*100
    changes = changes.dropna()
    median_change = changes.median()
    mad = (changes-median_change).abs().median()
    if mad == 0:
        return None
    scores = 0.6745 * (changes-median_change)/mad
    result = pd.DataFrame({"date":dates,"change":changes,"score":scores})
    result = result[result["score"].abs()>cutoff]
    return result

A stricter cutoff of 5 flags 6 weeks instead of 19, which shows the cutoff
input works.

In [12]:
find_jumps(data["total_reserves_usd"],data["week_ended"],5)

,date,change,score
99,2003-02-28,-3.259454,-5.337429
247,2005-12-30,-4.751128,-7.629843
338,2007-09-28,5.032409,7.405556
392,2008-10-10,-3.499671,-5.706595
394,2008-10-24,-5.648701,-9.009237
1326,2026-09-04,6.061359,8.986853


Now the same check on the rupee column, for comparison.

In [13]:
print(find_jumps(data["total_reserves_inr"],data["week_ended"]))

           date    change     score
99   2003-02-28 -3.407197 -4.519074
130  2003-10-03 -2.749874 -3.706622
247  2005-12-30 -4.997012 -6.484088
338  2007-09-28  4.606336  5.385665
371  2008-05-16  3.410841  3.908030
388  2008-09-12  3.293571  3.763085
394  2008-10-24 -3.272746 -4.352892
395  2008-10-31 -3.319125 -4.410217
396  2008-11-07 -3.507424 -4.642955
476  2010-05-21  3.852017  4.453325
562  2012-01-13 -2.660871 -3.596614
646  2013-08-23  3.890576  4.500984
649  2013-09-13 -2.837439 -3.814852
987  2020-03-06  3.378527  3.868090
1232 2024-11-15 -2.597352 -3.518105
1326 2026-09-04  5.063419  5.950620


In [14]:
usd_flags = find_jumps(data["total_reserves_usd"],data["week_ended"])
inr_flags = find_jumps(data["total_reserves_inr"],data["week_ended"])
both = set(usd_flags["date"]) & set(inr_flags["date"])
print(len(usd_flags),len(inr_flags),len(both))
print("rupee only :",sorted(set(inr_flags["date"]) - set(usd_flags["date"])))

19 16 7
rupee only : [Timestamp('2003-10-03 00:00:00'), Timestamp('2008-05-16 00:00:00'), Timestamp('2008-09-12 00:00:00'), Timestamp('2008-11-07 00:00:00'), Timestamp('2010-05-21 00:00:00'), Timestamp('2012-01-13 00:00:00'), Timestamp('2013-08-23 00:00:00'), Timestamp('2013-09-13 00:00:00'), Timestamp('2020-03-06 00:00:00')]


On dollar reserves the check flags 19 weeks. The rupee column flags 16, and
only 7 weeks are in both lists. Several rupee-only flags are exchange-rate
moves, like 2010-05-21. And some real drops in autumn 2008 show up only in
the dollar column, because the falling rupee cushioned them in rupee terms.